# 🛡️ RiskRadar — EDA & Modelling**Predicting women's-safety risk from urban signals.**This notebook is the analytical companion to the production package in `src/riskradar/`.It imports the *same* code the API serves, so nothing here can drift from what actually ships.| | ||---|---|| Dataset | 100,000 area-hour records || Target | `Risk_Level` ∈ {Low, Medium, High} || Result | 98.27% accuracy, 0.9993 ROC-AUC || Previous version | 94.83% |

## 1 · Setup

In [ ]:
import sys, json, warningsfrom pathlib import PathROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT / "src"))warnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom riskradar import config as C, data as Dfrom riskradar.features import engineer, feature_namessns.set_theme(style="darkgrid")plt.rcParams.update({    "figure.facecolor": "#0b0f1e", "axes.facecolor": "#0b0f1e",    "axes.edgecolor": "#2a3350", "grid.color": "#1b2238",    "text.color": "#EEF0F8", "axes.labelcolor": "#B4BAD0",    "xtick.color": "#79809B", "ytick.color": "#79809B",    "figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12,})PALETTE = {"Low": "#2DD4A7", "Medium": "#FBBF24", "High": "#FB4E6D"}print("Environment ready")

## 2 · Load and audit the dataBefore any modelling, prove the data is clean.

In [ ]:
df = D.load_dataset()quality = D.data_quality_report(df)print(f"Shape          : {df.shape}")print(f"Missing values : {quality['missing_values']}")print(f"Duplicate rows : {quality['duplicate_rows']}")print(f"Constant cols  : {quality['constant_columns']}")print(f"Class balance  : {quality['class_distribution']}")print(f"Imbalance ratio: {quality['imbalance_ratio']}x")df.head()

### A subtlety worth catching early`Crime_Count` is *not* the sum of the itemised crime categories. Only ~7.5% of rows add up,so the residual carries real information — captured later as `Unclassified_Crime`.

In [ ]:
itemised = df[["Violent_Crime", "Theft_Count", "Assault_Count", "Harassment_Count"]].sum(axis=1)print(f"Rows where Crime_Count == sum(parts): {(df.Crime_Count == itemised).mean():.2%}")print(f"Rows where Streetlights == working + broken: {(df.Streetlight_Count == df.Working_Streetlights + df.Broken_Streetlights).mean():.2%}")

## 3 · Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))order = ["Low", "Medium", "High"]counts = df.Risk_Level.value_counts().reindex(order)axes[0].bar(order, counts.values, color=[PALETTE[c] for c in order])axes[0].set_title("Risk level distribution")for i, v in enumerate(counts.values):    axes[0].text(i, v * 1.01, f"{v:,}", ha="center", fontsize=9)hourly = df.groupby("Hour").Risk_Level.apply(lambda s: (s == "High").mean())axes[1].fill_between(hourly.index, hourly.values * 100, color="#FB4E6D", alpha=.28)axes[1].plot(hourly.index, hourly.values * 100, color="#FB4E6D", lw=2.2)axes[1].set_title("High-risk share by hour")axes[1].set_xlabel("Hour of day"); axes[1].set_ylabel("% high risk")plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))light_ratio = df.Working_Streetlights / (df.Streetlight_Count + 1)bands = pd.cut(light_ratio, [0, .6, .75, .85, 1.01], labels=["<60%", "60-75%", "75-85%", "85%+"])by_light = df.assign(b=bands).groupby("b", observed=True).Risk_Level.apply(lambda s: (s == "High").mean())axes[0].bar(by_light.index.astype(str), by_light.values * 100, color="#2DD4A7")axes[0].set_title("Working-streetlight ratio vs risk"); axes[0].set_ylabel("% high risk")pol = pd.qcut(df.Police_Distance_km, 5)by_pol = df.assign(b=pol).groupby("b", observed=True).Risk_Level.apply(lambda s: (s == "High").mean())axes[1].bar(range(5), by_pol.values * 100, color="#8B7CFF")axes[1].set_xticks(range(5)); axes[1].set_xticklabels([f"Q{i+1}" for i in range(5)])axes[1].set_title("Police distance quintile vs risk")persist = pd.crosstab(df.Previous_Risk, df.Risk_Level, normalize="index").reindex(order)[order]sns.heatmap(persist, annot=True, fmt=".2f", cmap="magma", ax=axes[2], cbar=False)axes[2].set_title("Previous risk → current risk")plt.tight_layout(); plt.show()

**Reading the plots**- Risk concentrates sharply overnight — the strongest single temporal signal.- Areas with under 60% of streetlights working are markedly more dangerous than those above 85%.- High-risk share rises monotonically with distance to police.- Risk persists: the previous → current crosstab is strongly diagonal, so history is predictive.

## 4 · Feature engineering29 raw columns → 79 features. Three rules, all enforced by tests in `tests/test_features.py`:1. **No target usage** — nothing reads `Risk_Level`.2. **No cross-row statistics** — every feature is a pure row-wise function, so there is no   train/test leakage and a single record can be scored in isolation by the API.3. **Exposure normalisation** — 40 incidents means something different in a dense commercial hub   than in a quiet residential lane.

In [ ]:
X, y = D.split_xy(df)          # Area_ID dropped: unique per row → memorisation riskF = engineer(X)print(f"Raw features       : {X.shape[1]}")print(f"Engineered features: {F.shape[1]}")print(f"Any NaN or inf     : {not np.isfinite(F.to_numpy()).all()}")F.iloc[:3, :12]

### The key interaction: darkness only matters at night

In [ ]:
probe = X.head(1).copy()probe["Broken_Streetlights"], probe["Working_Streetlights"], probe["Streetlight_Count"] = 30, 10, 40probe["Visibility"] = "Poor"for hour in [2, 8, 14, 20, 23]:    val = engineer(probe.assign(Hour=hour))["Darkness_Exposure"].iloc[0]    print(f"  {hour:02d}:00 → Darkness_Exposure = {val:.3f}")

### How much does engineering actually buy?Compare the *same* model on raw vs engineered features.

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import make_pipeline as mkfrom sklearn.metrics import accuracy_scoreX_tr, X_te, y_tr, y_te = D.stratified_split(X, y)raw_tr, raw_te = X_tr.copy(), X_te.copy()for col in raw_tr.select_dtypes("object"):    cats = sorted(set(raw_tr[col]) | set(raw_te[col]))    m = {c: i for i, c in enumerate(cats)}    raw_tr[col], raw_te[col] = raw_tr[col].map(m), raw_te[col].map(m)baseline = mk(StandardScaler(), LogisticRegression(max_iter=1200)).fit(raw_tr, y_tr)improved = mk(StandardScaler(), LogisticRegression(max_iter=1200)).fit(engineer(X_tr), y_tr)a = accuracy_score(y_te, baseline.predict(raw_te))b = accuracy_score(y_te, improved.predict(engineer(X_te)))print(f"Logistic regression, raw features       : {a:.2%}")print(f"Logistic regression, engineered features: {b:.2%}")print(f"Gain from feature engineering alone     : +{(b - a) * 100:.2f} points")

## 5 · Model benchmarkNine candidates on an identical stratified split with identical features. Every candidate is afull `Pipeline` whose first step is the feature transform, so the saved artefact accepts **raw**records and the API never re-implements preprocessing.Run the full suite from the command line (it takes several minutes):```bashpython -m riskradar.train```Below we load the results it produced.

In [ ]:
leaderboard = pd.DataFrame(json.loads((ROOT / "reports" / "model_leaderboard.json").read_text()))view = leaderboard[["model", "accuracy", "f1_macro", "roc_auc_ovr", "log_loss",                    "cohen_kappa", "cv_accuracy_mean", "fit_seconds"]].copy()for c in ["accuracy", "f1_macro"]:    view[c] = (view[c] * 100).round(2)view

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))board = leaderboard[leaderboard.accuracy > .5].sort_values("accuracy")colors = ["#8B7CFF" if m == board.model.iloc[-1] else "#3a3f66" for m in board.model]ax.barh(board.model, board.accuracy * 100, color=colors)ax.axvline(94.83, ls="--", color="#FBBF24", lw=1.6, label="previous project (94.83%)")ax.set_xlim(85, 100); ax.set_xlabel("Accuracy (%)"); ax.set_title("Model leaderboard")for i, v in enumerate(board.accuracy * 100):    ax.text(v + .12, i, f"{v:.2f}", va="center", fontsize=8.5)ax.legend(); plt.tight_layout(); plt.show()

## 6 · Final model evaluation

In [ ]:
import joblibmodel = joblib.load(ROOT / "models" / "riskradar_model.joblib")meta = json.loads((ROOT / "models" / "model_metadata.json").read_text())met = meta["metrics"]print(f"Deployed model : {meta['model_name']}")print(f"Members        : {meta['ensemble_members']}")print()for k in ["accuracy", "balanced_accuracy", "f1_macro", "roc_auc_ovr",          "log_loss", "cohen_kappa", "mcc", "expected_calibration_error"]:    print(f"  {k:28s} {met[k]:.4f}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplaycm = np.array(met["confusion_matrix"])fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))sns.heatmap(cm, annot=True, fmt="d", cmap="mako", cbar=False,            xticklabels=C.CLASS_ORDER, yticklabels=C.CLASS_ORDER, ax=axes[0])axes[0].set_title("Confusion matrix"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")imp = pd.DataFrame(meta["feature_importance"][:15])axes[1].barh(imp.feature[::-1], imp.importance[::-1] * 100, color="#22D3EE")axes[1].set_title("Top 15 features"); axes[1].set_xlabel("Importance (%)")plt.tight_layout(); plt.show()print("Low↔High confusions:", cm[0, 2] + cm[2, 0], "— the catastrophic error mode never occurred.")

### Error profileThe two corner cells are **zero**. The model never called a genuinely High-risk area "Low", nor aLow-risk area "High". Every error is a single-band slip adjacent to Medium — the least costlymistake this system can make, and exactly the profile you want in a safety application.

## 7 · ExplainabilityExact Shapley values via `TreeExplainer`, plus a counterfactual scan for interventions.

In [ ]:
from riskradar.service import RiskServicesvc = RiskService.instance()risky = {    "Latitude": 13.19, "Longitude": 80.04, "Crime_Count": 71, "Violent_Crime": 16,    "Theft_Count": 29, "Assault_Count": 14, "Harassment_Count": 12, "Emergency_Calls": 41,    "Hour": 2, "Day": "Sun", "Month": 11, "Weekend": 1,    "Streetlight_Count": 50, "Working_Streetlights": 21, "Broken_Streetlights": 29,    "CCTV_Count": 4, "Population_Density": 8800, "Footfall": 140,    "Bus_Stop_Count": 2, "Metro_Distance_km": 11.2, "Police_Distance_km": 9.3,    "Hospital_Distance_km": 7.8, "School_Count": 0,    "Commercial_Area": 0, "Residential_Area": 1,    "Weather": "Fog", "Visibility": "Poor",    "Previous_Risk": "High", "Crime_Trend": "Increasing",}result = svc.predict(risky)print(f"Prediction : {result['risk_level']} ({result['confidence']:.1%} confidence)")print(f"Safety score: {result['safety_score']}/100")print(f"Method     : {result['explanation']['method']}")print(f"\n{result['explanation']['narrative']}\n")for d in result["explanation"]["drivers"][:8]:    arrow = "▲" if d["impact"] > 0 else "▼"    print(f"  {arrow} {d['label']:36s} {d['impact']:+.3f}  (value {d['value']})")

In [ ]:
def show_interventions(name, payload):    r = svc.predict(payload)    print(f"{name}: predicted {r['risk_level']} (safety {r['safety_score']})")    if r["recommendations"]:        for rec in r["recommendations"]:            print(f"   • {rec['action']}: {rec['detail']} → falls to {rec['new_risk']}")    else:        print("   • no single lever is sufficient — this area needs combined investment")    print()show_interventions("Extreme case", risky)moderate = risky | {    "Crime_Count": 34, "Violent_Crime": 6, "Theft_Count": 17, "Assault_Count": 5,    "Harassment_Count": 6, "Emergency_Calls": 15, "Hour": 21,    "Working_Streetlights": 40, "Broken_Streetlights": 15, "Streetlight_Count": 55,    "CCTV_Count": 25, "Police_Distance_km": 3.5, "Footfall": 1400,    "Previous_Risk": "Medium", "Crime_Trend": "Stable", "Visibility": "Medium",}show_interventions("Moderate case", moderate)

Note the honesty of the first result: an area with 71 incidents, 29 broken lamps, four cameras andpolice 9.3 km away cannot be fixed by any *single* lever, and the system says so rather thaninventing a reassuring answer. The moderate case is where intervention advice becomes actionable.

### Sensitivity: how does risk respond to surveillance coverage?

In [ ]:
sweep = svc.what_if(moderate, "CCTV_Count", [0, 25, 50, 75, 100, 150])for v, lvl, sc in zip(sweep["values"], sweep["risk_levels"], sweep["safety_scores"]):    bar = "█" * int(sc / 4)    print(f"  {int(v):3d} cameras → {lvl:6s}  safety {sc:5.1f}  {bar}")fig, ax = plt.subplots(figsize=(7, 3.4))ax.plot(sweep["values"], sweep["safety_scores"], color="#22D3EE", lw=2.4, marker="o")ax.fill_between(sweep["values"], sweep["safety_scores"], color="#22D3EE", alpha=.2)ax.set_xlabel("CCTV cameras"); ax.set_ylabel("Safety score")ax.set_title("Risk response to surveillance coverage")plt.tight_layout(); plt.show()

## 8 · Conclusions| | v1 | v2 | Change ||---|---:|---:|---:|| Accuracy | 94.83% | **98.27%** | +3.44 pts || Error rate | 5.17% | **1.73%** | −66% || Macro F1 | — | 97.68% | — || Calibration (ECE) | — | 0.0015 | — || Tests | 0 | 62 | — |**Findings**1. Feature engineering did the heavy lifting — logistic regression on the engineered features   (97.44%) beats every tree model trained on the raw ones.2. Narrow trees generalise better here: 31 leaves outperformed 127 and 255, so the underlying   signal is smooth and additive rather than deeply interactive.3. Ensemble membership must be earned. A fixed five-member vote scored *worse* than its best   member; filtering by tolerance and sharpening the weights recovered the full accuracy.4. The error profile is the right shape — zero Low↔High confusions in 20,000 predictions.**Limitations.** The corpus is synthetic; reported crime is a biased proxy for actual crime; andmodel selection on hold-out accuracy introduces mild optimism (cross-validation, 98.00%, is themore conservative estimate). See `docs/MODEL_CARD.md` for the full disclosure.